In [1]:
import copy
import pickle

from pyflink.common import Row, Types
from pyflink.datastream import (
    KeyedProcessFunction,
    RuntimeContext,
    StreamExecutionEnvironment,
)
from pyflink.datastream.state import ValueStateDescriptor
from pyflink.table import EnvironmentSettings, StreamTableEnvironment

In [2]:
# --- 1️⃣ Environment setup ---
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(4)
settings = EnvironmentSettings.new_instance().in_streaming_mode().build()
t_env = StreamTableEnvironment.create(env, environment_settings=settings)

/usr/local/lib/python3.11/dist-packages/apache_beam/runners/portability/stager.py:63: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
# --- 2️⃣ Kafka source ---
t_env.execute_sql("DROP TABLE IF EXISTS aggtrades_source")
t_env.execute_sql("""
CREATE TABLE aggtrades_source (
  agg_trade_id BIGINT,
  price DOUBLE,
  quantity DOUBLE,
  first_trade_id BIGINT,
  last_trade_id BIGINT,
  ts_int BIGINT,
  is_buyer_maker BOOLEAN,
  is_best_match BOOLEAN,
  symbol STRING
) WITH (
  'connector' = 'kafka',
  'topic' = 'aggtrades-topic',
  'properties.bootstrap.servers' = 'kafka:9092',
  'properties.group.id' = 'pyflink_aggtrades_consumer',
  'scan.startup.mode' = 'earliest-offset',
  'format' = 'json',
  'json.ignore-parse-errors' = 'true'
)
""")

In [4]:
result = t_env.execute_sql("SELECT * FROM aggtrades_source LIMIT 20")
for row in result.collect():
    print(row)

<Row(411025109, 0.7637, 245.5, 711415900, 711415900, 1758844800664531, True, True, 'ADAUSDT')>
<Row(411025110, 0.7638, 44.3, 711415901, 711415901, 1758844800879844, False, True, 'ADAUSDT')>
<Row(411025111, 0.7638, 130.9, 711415902, 711415909, 1758844801316853, False, True, 'ADAUSDT')>
<Row(411025112, 0.7638, 39.2, 711415910, 711415911, 1758844801437467, False, True, 'ADAUSDT')>
<Row(411025113, 0.7638, 1055.8, 711415912, 711415912, 1758844801516590, False, True, 'ADAUSDT')>
<Row(411025114, 0.7638, 20.7, 711415913, 711415915, 1758844802674185, False, True, 'ADAUSDT')>
<Row(411025115, 0.7639, 46.3, 711415916, 711415920, 1758844802680605, False, True, 'ADAUSDT')>
<Row(411025116, 0.764, 5691.9, 711415921, 711415930, 1758844803218192, False, True, 'ADAUSDT')>
<Row(411025117, 0.764, 2231.3, 711415931, 711415931, 1758844803223545, False, True, 'ADAUSDT')>
<Row(411025118, 0.764, 65.3, 711415932, 711415932, 1758844804009679, False, True, 'ADAUSDT')>
<Row(411025119, 0.764, 98978.1, 711415933, 711

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/py4j/java_gateway.py", line 1217, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3699, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_16/1598937350.py", line 2, in <module>
    for row in result.collect():
  File "/usr/local/lib/python3.11

Unexpected exception formatting exception. Falling back to standard exception


In [4]:
# --- 3️⃣ klines view ---
t_env.execute_sql("DROP TEMPORARY VIEW IF EXISTS klines_view")
t_env.execute_sql("""
CREATE TEMPORARY VIEW klines_view AS
SELECT
    window_start,
    window_end,
    ROUND(FIRST_VALUE(price), 4) AS open_price,
    ROUND(MAX(price), 4) AS high_price,
    ROUND(MIN(price), 4) AS low_price,
    ROUND(LAST_VALUE(price), 4) AS close_price,
    ROUND(SUM(quantity), 1) AS volume,
    'ADAUSDT' AS symbol
FROM TABLE(
    TUMBLE(TABLE aggtrades_source, DESCRIPTOR(ts), INTERVAL '15' MINUTES)
)
GROUP BY window_start, window_end
""")

In [5]:
# --- 4️⃣ ClickHouse sinks ---
t_env.execute_sql("DROP TABLE IF EXISTS klines")
t_env.execute_sql("""
CREATE TABLE klines (
    window_start TIMESTAMP(3),
    window_end TIMESTAMP(3),
    open_price DOUBLE,
    high_price DOUBLE,
    low_price DOUBLE,
    close_price DOUBLE,
    volume DOUBLE,
    symbol STRING
) WITH (
    'connector' = 'clickhouse',
    'url' = 'jdbc:clickhouse://clickhouse:8123',
    'database-name' = 'testdb',
    'table-name' = 'klines',
    'username' = 'default',
    'password' = '123456',
    'sink.batch-size' = '5000',
    'sink.flush-interval' = '2s',
    'sink.max-retries' = '3',
    'sink.parallelism' = '1'
)
""")

In [6]:
t_env.execute_sql("DROP TABLE IF EXISTS engulfing_clickhouse")
t_env.execute_sql("""
CREATE TABLE engulfing_clickhouse (
    window_start TIMESTAMP(3),
    window_end TIMESTAMP(3),
    open_price DOUBLE,
    high_price DOUBLE,
    low_price DOUBLE,
    close_price DOUBLE,
    volume DOUBLE,
    ema7 DOUBLE,
    ema20 DOUBLE,
    trend STRING,
    engulfing_pattern STRING,
    symbol STRING
) WITH (
    'connector' = 'clickhouse',
    'url' = 'jdbc:clickhouse://clickhouse:8123',
    'database-name' = 'testdb',
    'table-name' = 'engulfings',
    'username' = 'default',
    'password' = '123456',
    'sink.batch-size' = '5000',
    'sink.flush-interval' = '2s',
    'sink.max-retries' = '3',
    'sink.parallelism' = '1'
)
""")

In [7]:
# --- 5️⃣ Kafka sink for engulfing (real-time events) ---
t_env.execute_sql("DROP TABLE IF EXISTS engulfing_kafka")
t_env.execute_sql("""
CREATE TABLE engulfing_kafka (
    window_start TIMESTAMP(3),
    window_end TIMESTAMP(3),
    open_price DOUBLE,
    high_price DOUBLE,
    low_price DOUBLE,
    close_price DOUBLE,
    volume DOUBLE,
    ema7 DOUBLE,
    ema20 DOUBLE,
    trend STRING,
    engulfing_pattern STRING,
    symbol STRING
) WITH (
    'connector' = 'kafka',
    'topic' = 'engulfings-topic',
    'properties.bootstrap.servers' = 'kafka:9092',
    'format' = 'json',
    'json.timestamp-format.standard' = 'ISO-8601'
)
""")

In [8]:
# --- 6️⃣ Utility functions ---
def round_half_up(x, decimals=2):
    if x is None:
        return None
    factor = 10**decimals
    return float(int(x * factor + 0.5)) / factor


# --- 7️⃣ Engulfing pattern function ---
class EngulfingPatternFunction(KeyedProcessFunction):
    def open(self, runtime_context: RuntimeContext):
        self.prev_row_state = runtime_context.get_state(
            ValueStateDescriptor("prev_row_state", Types.PICKLED_BYTE_ARRAY())
        )

    def calc_ema(self, close_price, period, ema_state, buffer_state):
        if close_price is None:
            return None
        k = 2 / (period + 1)
        if ema_state is None:
            buffer_state.append(close_price)
            if len(buffer_state) < period:
                return None
            ema = sum(buffer_state) / len(buffer_state)
            buffer_state.clear()
            return ema
        else:
            return (close_price - ema_state) * k + ema_state

    def detect_trend(self, ema7, ema20):
        if ema7 is None or ema20 is None:
            return None
        if ema7 > ema20:
            return "uptrend"
        elif ema7 < ema20:
            return "downtrend"
        return None

    def detect_engulfing(self, current, previous, trend):
        if not previous or not trend:
            return None
        op_prev, cp_prev = previous["open_price"], previous["close_price"]
        op, cp = current["open_price"], current["close_price"]

        if (
            cp_prev < op_prev
            and cp > op
            and op < cp_prev
            and cp > op_prev
            and trend == "downtrend"
        ):
            return "bullish engulfing"
        if (
            cp_prev > op_prev
            and cp < op
            and op > cp_prev
            and cp < op_prev
            and trend == "uptrend"
        ):
            return "bearish engulfing"
        return None

    def process_element(self, value, ctx):
        prev_bytes = self.prev_row_state.value()
        prev_state = pickle.loads(prev_bytes) if prev_bytes else {}
        buf7 = copy.deepcopy(prev_state.get("buffer7_state", []))
        buf20 = copy.deepcopy(prev_state.get("buffer20_state", []))

        ema7 = self.calc_ema(value["close_price"], 7, prev_state.get("ema7"), buf7)
        ema20 = self.calc_ema(value["close_price"], 20, prev_state.get("ema20"), buf20)
        trend = self.detect_trend(ema7, ema20)
        pattern = self.detect_engulfing(value.as_dict(), prev_state, trend)

        new_state = {
            **value.as_dict(),
            "ema7": ema7,
            "ema20": ema20,
            "trend": trend,
            "pattern": pattern,
            "buffer7_state": buf7,
            "buffer20_state": buf20,
        }
        self.prev_row_state.update(pickle.dumps(new_state))

        yield Row(
            **value.as_dict(),
            ema7=round_half_up(ema7, 4) if ema7 else None,
            ema20=round_half_up(ema20, 4) if ema20 else None,
            trend=trend,
            engulfing_pattern=pattern,
        )

In [9]:
# --- 8️⃣ Output type info ---
typeinfo = Types.ROW_NAMED(
    [
        "window_start",
        "window_end",
        "open_price",
        "high_price",
        "low_price",
        "close_price",
        "volume",
        "ema7",
        "ema20",
        "trend",
        "engulfing_pattern",
        "symbol",
    ],
    [
        Types.SQL_TIMESTAMP(),
        Types.SQL_TIMESTAMP(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.STRING(),
        Types.STRING(),
        Types.STRING(),
    ],
)

In [10]:
# --- 9️⃣ Apply process ---
klines_stream = t_env.to_data_stream(t_env.from_path("klines_view"))
engulfing_stream = klines_stream.key_by(lambda x: x["symbol"]).process(
    EngulfingPatternFunction(), output_type=typeinfo
)
t_env.drop_temporary_view("engulfing_view")
t_env.create_temporary_view("engulfing_view", t_env.from_data_stream(engulfing_stream))

# --- 🔟 Execute 3 sinks ---
statement_set = t_env.create_statement_set()
statement_set.add_insert_sql("INSERT INTO klines SELECT * FROM klines_view")
statement_set.add_insert_sql("INSERT INTO engulfing_clickhouse SELECT * FROM engulfing_view")
statement_set.add_insert_sql("INSERT INTO engulfing_kafka SELECT * FROM engulfing_view")
statement_set.execute().wait()

Py4JJavaError: An error occurred while calling o430.execute.
: java.lang.RuntimeException: Build ClickHouse output format failed.
	at org.apache.flink.connector.clickhouse.internal.AbstractClickHouseOutputFormat$Builder.build(AbstractClickHouseOutputFormat.java:205)
	at org.apache.flink.connector.clickhouse.ClickHouseDynamicTableSink.getSinkRuntimeProvider(ClickHouseDynamicTableSink.java:83)
	at org.apache.flink.table.planner.plan.nodes.exec.common.CommonExecSink.createSinkTransformation(CommonExecSink.java:151)
	at org.apache.flink.table.planner.plan.nodes.exec.stream.StreamExecSink.translateToPlanInternal(StreamExecSink.java:214)
	at org.apache.flink.table.planner.plan.nodes.exec.ExecNodeBase.translateToPlan(ExecNodeBase.java:168)
	at org.apache.flink.table.planner.delegation.StreamPlanner.$anonfun$translateToPlan$1(StreamPlanner.scala:85)
	at scala.collection.TraversableLike.$anonfun$map$1(TraversableLike.scala:233)
	at scala.collection.Iterator.foreach(Iterator.scala:937)
	at scala

In [11]:
!jupyter nbconvert --to script end_transform_job_pattern_two.ipynb

[NbConvertApp] Converting notebook end_transform_job_pattern_two.ipynb to script
[NbConvertApp] Writing 8432 bytes to end_transform_job_pattern_two.py
